## Tools

Models can request to call tools that perform tasks such as fetching data from database, searching the web, or running code. Tools are pairing of:
    
1. A schema, including the name of the tool, a description, and/or argument definition (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response.content

'<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know. Parrots are known for their ability to mimic human speech. But why do they do that in the first place? I remember reading somewhere that it\'s part of their social behavior. Maybe they use it to communicate with each other? But then why would they mimic humans specifically?\n\nFirst, I should think about their natural environment. Parrots live in groups, right? So in the wild, they probably use vocalizations to communicate with their flock. That makes sense. They might mimic sounds to bond with each other or to establish social hierarchies. But how does that translate to talking with humans? Maybe when they\'re in captivity or living with humans, they adapt their mimicry skills to include human words. \n\nI also remember that some parrots can learn a lot of words. But not all parrots talk. So maybe it\'s a combination of the species and the environment. For example, African Greys are known 

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str)->str:
    """Get the current weather for a given location."""
    # In a real implementation, this would call a weather API.
    return f"The current weather in {location} is sunny with a high of 25°C."

model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What is the weather like in New York?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

### Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in New York is sunny with a high of 25°C."

The current weather in New York is sunny with a high of 25°C. A pleasant day to enjoy outdoor activities! 🌞


In [5]:
messages

[{'role': 'user', 'content': 'What is the weather like in New York?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. The user mentioned "New York," so I\'ll plug that into the location field. I\'ll make sure the JSON is correctly formatted with the name of the function and the arguments. No other parameters are needed here. Just call get_weather with location set to New York.\n', 'tool_calls': [{'id': '5rzggsr00', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 157, 'total_tokens': 277, 'completion_time': 0.203119492, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.007359328, 'prompt_tokens_details': None, 'queue_time': 0.05